In [ ]:
from pathlib import Path

bank_path = Path('/content/bank_selected.parquet')
telecom_path = Path('/content/telecom_train.parquet')

for path in [bank_path, telecom_path]:
    print(f'Файл: {path.name}')
    print(f'Найден: {path.exists()}')

    if path.exists():
        print(f'Размер: {path.stat().st_size / 1024**2:.2f} МБ')
    print()


Файл: bank_selected.parquet
Найден: True
Размер: 128.83 МБ

Файл: telecom_train.parquet
Найден: True
Размер: 2.38 МБ



In [ ]:

import pyarrow.parquet as pq
import pandas as pd

bank_file = pq.ParquetFile(bank_path)
telecom_file = pq.ParquetFile(telecom_path)

def show_structure(parquet_file, name):
    print(f'\n===== {name} =====')
    print(f'Строк: {parquet_file.metadata.num_rows:,}')
    print(f'Столбцов: {parquet_file.metadata.num_columns}')

    schema = parquet_file.schema_arrow

    info = pd.DataFrame({
        'Название столбца': schema.names,
        'Тип данных': [str(field.type) for field in schema]
    })

    display(info)

show_structure(bank_file, 'БАНК')
show_structure(telecom_file, 'ТЕЛЕКОМ')



===== БАНК =====
Строк: 307,511
Столбцов: 382


,Название столбца,Тип данных
0,SK_ID_CURR,int64
1,TARGET,int64
2,CODE_GENDER,string
3,AMT_GOODS_PRICE,double
4,DAYS_BIRTH,int64
...,...,...
377,CC_SK_DPD_MAX,double
378,CC_NAME_CONTRACT_STATUS_COMPLETED_MEAN,float
379,PREV_NAME_GOODS_CATEGORY_VEHICLES_MEAN,float
380,BUREAU_BB_STATUS_3_MEAN_MEAN,float



===== ТЕЛЕКОМ =====
Строк: 307,511
Столбцов: 6


,Название столбца,Тип данных
0,SK_ID_CURR,int64
1,OBS_30_CNT_SOCIAL_CIRCLE,double
2,DEF_30_CNT_SOCIAL_CIRCLE,double
3,OBS_60_CNT_SOCIAL_CIRCLE,double
4,DEF_60_CNT_SOCIAL_CIRCLE,double
5,DAYS_LAST_PHONE_CHANGE,double


In [ ]:
def show_first_rows(parquet_file, n=5):
    batches = parquet_file.iter_batches(batch_size=n)
    first_batch = next(batches, None)

    if first_batch is None:
        return pd.DataFrame()

    return first_batch.to_pandas().head(n)

print("БАНК:")
display(show_first_rows(bank_file))

print("ТЕЛЕКОМ:")
display(show_first_rows(telecom_file))


БАНК:


,SK_ID_CURR,TARGET,CODE_GENDER,AMT_GOODS_PRICE,DAYS_BIRTH,BUREAU_DAYS_CREDIT_MEAN,AMT_CREDIT,INST_AMT_PAYMENT_MIN,DAYS_EMPLOYED,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,...,PREV_NAME_GOODS_CATEGORY_CONSTRUCTION_MATERIALS_MEAN,FLAG_DOCUMENT_6,BUREAU_AMT_ANNUITY_MIN,BUREAU_AMT_CREDIT_SUM_LIMIT_MIN,PREV_NAME_SELLER_INDUSTRY_INDUSTRY_MEAN,CC_SK_DPD_MAX,CC_NAME_CONTRACT_STATUS_COMPLETED_MEAN,PREV_NAME_GOODS_CATEGORY_VEHICLES_MEAN,BUREAU_BB_STATUS_3_MEAN_MEAN,PREV_CODE_REJECT_REASON_XNA_MEAN
0,100002,1,M,351000.0,-9461,-874.00,406597.5,9251.775,-637,49156.2,...,0.000000,0,0.0,0.0,0.0,NaN,NaN,1.0,0.0,0.0
1,100003,0,F,1129500.0,-16765,-1400.75,1293502.5,6662.970,-1188,0.0,...,0.000000,0,NaN,0.0,0.0,NaN,NaN,0.0,NaN,0.0
2,100004,0,M,135000.0,-19046,-867.00,135000.0,5357.250,-225,0.0,...,0.000000,0,NaN,0.0,0.0,NaN,NaN,0.0,NaN,0.0
3,100006,0,F,297000.0,-19005,NaN,312682.5,2482.920,-3039,NaN,...,0.111111,0,NaN,NaN,0.0,0.0,0.0,0.0,NaN,0.0
4,100007,0,M,513000.0,-19932,-1149.00,513000.0,0.180,-3038,0.0,...,0.000000,0,NaN,0.0,0.0,NaN,NaN,0.0,NaN,0.0


ТЕЛЕКОМ:


,SK_ID_CURR,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE
0,100002,2.0,2.0,2.0,2.0,-1134.0
1,100003,1.0,0.0,1.0,0.0,-828.0
2,100004,0.0,0.0,0.0,0.0,-815.0
3,100006,2.0,0.0,2.0,0.0,-617.0
4,100007,0.0,0.0,0.0,0.0,-1106.0


In [ ]:
import pandas as pd

# Загружаем только идентификаторы
bank_ids = pd.read_parquet(
    '/content/bank_selected.parquet',
    columns=['SK_ID_CURR']
)

telecom_ids = pd.read_parquet(
    '/content/telecom_train.parquet',
    columns=['SK_ID_CURR']
)

# Преобразуем в множества
X = set(bank_ids['SK_ID_CURR'].dropna())
Y = set(telecom_ids['SK_ID_CURR'].dropna())

# Находим пересечение для проверки данных
intersection = X.intersection(Y)

print("Клиентов банка:", len(X))
print("Клиентов телекома:", len(Y))
print("Общих клиентов:", len(intersection))

print("\nТолько у банка:", len(X - Y))
print("Только у телекома:", len(Y - X))

print("\nДоля совпадений среди клиентов банка:",
      f"{len(intersection) / len(X) * 100:.2f}%")

print("Доля совпадений среди клиентов телекома:",
      f"{len(intersection) / len(Y) * 100:.2f}%")


Клиентов банка: 307511
Клиентов телекома: 307511
Общих клиентов: 307511

Только у банка: 0
Только у телекома: 0

Доля совпадений среди клиентов банка: 100.00%
Доля совпадений среди клиентов телекома: 100.00%


In [ ]:
# X и Y уже содержат идентификаторы из предыдущего шага

all_ids = sorted(X & Y)

# 100 клиентов банка
bank_test = [str(x) for x in all_ids[:100]]

# 50 общих клиентов + 50 других клиентов телекома
telecom_test = (
    [str(x) for x in all_ids[50:100]]
    + [str(x) for x in all_ids[100:150]]
)

# Эталонное пересечение (только для тестирования)
expected = set(bank_test) & set(telecom_test)

print("Банк:", len(bank_test))
print("Телеком:", len(telecom_test))
print("Общих клиентов:", len(expected))
print("Первые 10 совпадений:", sorted(expected)[:10])


Банк: 100
Телеком: 100
Общих клиентов: 50
Первые 10 совпадений: ['100058', '100059', '100060', '100061', '100062', '100063', '100064', '100068', '100069', '100070']


In [ ]:
%pip install -q openmined-psi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.30.2 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 6.30.2 which is incompatible.
grpcio-status 1.85.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.30.2 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.30.2 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.30.2 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.30.2 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 6.30.2 which is incompatible.
google-cloud-mon

In [ ]:

import sys
import importlib.util
from importlib import metadata

print("Используемый Python:")
print(sys.executable)

print("\nВерсия Python:")
print(sys.version)

print("\nИнформация об установленном пакете:")

try:
    dist = metadata.distribution("openmined-psi")

    print("Версия:", dist.version)
    print("Расположение:", dist.locate_file(""))

    print("\nФайлы пакета:")
    for f in list(dist.files or [])[:30]:
        print(f)

except metadata.PackageNotFoundError:
    print("Пакет не найден в текущем окружении")

print("\nПроверка модулей:")
for name in ["psi", "openmined_psi"]:
    print(name, ":", importlib.util.find_spec(name))


Используемый Python:
/usr/bin/python3

Версия Python:
3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]

Информация об установленном пакете:
Версия: 2.0.6
Расположение: /usr/local/lib/python3.13/dist-packages

Файлы пакета:
openmined_psi-2.0.6.dist-info/INSTALLER
openmined_psi-2.0.6.dist-info/LICENSE
openmined_psi-2.0.6.dist-info/METADATA
openmined_psi-2.0.6.dist-info/README
openmined_psi-2.0.6.dist-info/RECORD
openmined_psi-2.0.6.dist-info/REQUESTED
openmined_psi-2.0.6.dist-info/WHEEL
private_set_intersection/proto/psi_python_proto_pb/private_set_intersection/proto/__pycache__/psi_pb2.cpython-313.pyc
private_set_intersection/proto/psi_python_proto_pb/private_set_intersection/proto/psi_pb2.py
private_set_intersection/python/__init__.py
private_set_intersection/python/__pycache__/__init__.cpython-313.pyc
private_set_intersection/python/_openmined_psi.so
site-packages/__init__.py
site-packages/__pycache__/__init__.cpython-313.pyc
site-packages/google/__init__.py
site-packages/google/__p

In [ ]:
import sys
import private_set_intersection.python as psi

print("Версия Python:", sys.version)
print("Библиотека PSI импортирована успешно!")

print("\nДоступные компоненты:")
print(dir(psi))

Версия Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Библиотека PSI импортирована успешно!

Доступные компоненты:
['DataStructure', 'Enum', 'List', 'Request', 'Response', 'ServerSetup', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_openmined_psi', 'client', 'psi', 'server']


In [ ]:
import private_set_intersection.python as psi

print("Методы клиента:")
print(dir(psi.client))

print("\nМетоды сервера:")
print(dir(psi.server))

print("\nДоступные структуры данных:")
print(list(psi.DataStructure))


Методы клиента:
['CreateFromKey', 'CreateRequest', 'CreateWithNewKey', 'GetIntersection', 'GetIntersectionSize', 'GetPrivateKeyBytes', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__']

Методы сервера:
['CreateFromKey', 'CreateSetupMessage', 'CreateWithNewKey', 'GetPrivateKeyBytes', 'ProcessRequest', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_a

In [ ]:
import private_set_intersection.python as psi

# Создаём клиента (банк) и сервер (телеком)
bank_client = psi.client.CreateWithNewKey(True) #внутри генерируется секретный ключ
telecom_server = psi.server.CreateWithNewKey(True)

# Телеком создаёт защищённое представление своей базы
server_setup = telecom_server.CreateSetupMessage(
    0.001,
    len(bank_test), #Ожидаемый размер множества клиента
    telecom_test, #Идентификаторы телекома (y)
    psi.DataStructure.RAW
) # в результате зашифрованные значения

# Банк формирует криптографический запрос
request = bank_client.CreateRequest(bank_test)

# Телеком обрабатывает запрос
response = telecom_server.ProcessRequest(request)
#телеком применяет свой секретный ключ к зашифрованным банковским значениям

# Банк вычисляет пересечение
indices = bank_client.GetIntersection(
    server_setup,
    response
)

# Преобразуем индексы в исходные ID банка
bank_result = [bank_test[i] for i in indices]

print("Найдено общих клиентов:", len(bank_result))
print("Первые 10 совпадений:", bank_result[:10])

# Проверяем результат
assert set(bank_result) == expected

print("PSI успешно выполнен!")


Найдено общих клиентов: 50
Первые 10 совпадений: ['100060', '100073', '100064', '100113', '100098', '100076', '100078', '100062', '100101', '100112']
PSI успешно выполнен!


In [ ]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd

bank_psi_path = Path('/content/bank_psi.parquet')
telecom_path = Path('/content/telecom_train.parquet')

# Проверяем файл
print("Файл найден:", bank_psi_path.exists())

if bank_psi_path.exists():
    bank_file = pq.ParquetFile(bank_psi_path)

    print("Количество строк:", bank_file.metadata.num_rows)
    print("Количество столбцов:", bank_file.metadata.num_columns)

    schema = bank_file.schema_arrow

    columns_info = pd.DataFrame({
        'Столбец': schema.names,
        'Тип данных': [str(field.type) for field in schema]
    })

    display(columns_info)


Файл найден: True
Количество строк: 307491
Количество столбцов: 382


,Столбец,Тип данных
0,SK_ID_CURR,int64
1,TARGET,int64
2,CODE_GENDER,string
3,AMT_GOODS_PRICE,double
4,DAYS_BIRTH,int64
...,...,...
377,CC_SK_DPD_MAX,double
378,CC_NAME_CONTRACT_STATUS_COMPLETED_MEAN,float
379,PREV_NAME_GOODS_CATEGORY_VEHICLES_MEAN,float
380,BUREAU_BB_STATUS_3_MEAN_MEAN,float


In [ ]:
# Загружаем только столбцы с идентификаторами
bank_ids = pd.read_parquet(
    bank_psi_path,
    columns=['SK_ID_CURR']
)

telecom_ids = pd.read_parquet(
    telecom_path,
    columns=['SK_ID_CURR']
)

# Формируем множества уникальных ID
X_new = set(bank_ids['SK_ID_CURR'].dropna().astype(str))
Y_new = set(telecom_ids['SK_ID_CURR'].dropna().astype(str))

# Вычисляем эталонное пересечение
expected_new = X_new & Y_new

print("Клиентов банка:", len(X_new))
print("Клиентов телекома:", len(Y_new))
print("Общих клиентов:", len(expected_new))

print("\nТолько у банка:", len(X_new - Y_new))
print("Только у телекома:", len(Y_new - X_new))

print("\nДоля общих клиентов среди клиентов банка:",
      f"{len(expected_new) / len(X_new) * 100:.2f}%"
      if X_new else "Нет клиентов")

print("Доля общих клиентов среди клиентов телекома:",
      f"{len(expected_new) / len(Y_new) * 100:.2f}%"
      if Y_new else "Нет клиентов")


Клиентов банка: 307491
Клиентов телекома: 307511
Общих клиентов: 307491

Только у банка: 0
Только у телекома: 20

Доля общих клиентов среди клиентов банка: 100.00%
Доля общих клиентов среди клиентов телекома: 99.99%


In [ ]:

import private_set_intersection.python as psi
import time

# Преобразуем множества в отсортированные списки
bank_new = sorted(X_new)
telecom_new = sorted(Y_new)

print("Начинаем PSI...")
print("Банк:", len(bank_new))
print("Телеком:", len(telecom_new))

start = time.perf_counter()

# Создаём участников с новыми секретными ключами
bank_client = psi.client.CreateWithNewKey(True)
telecom_server = psi.server.CreateWithNewKey(True)

# Телеком подготавливает защищённую базу
server_setup = telecom_server.CreateSetupMessage(
    0.001,
    len(bank_new),
    telecom_new,
    psi.DataStructure.RAW
)

# Банк формирует запрос
request = bank_client.CreateRequest(bank_new)

# Телеком применяет свой секретный ключ
response = telecom_server.ProcessRequest(request)

# Банк вычисляет пересечение
indices = bank_client.GetIntersection(
    server_setup,
    response
)

# Получаем идентификаторы совпавших клиентов
bank_result_new = [bank_new[i] for i in indices]

elapsed = time.perf_counter() - start

# Проверяем корректность
assert set(bank_result_new) == expected_new

print("\nPSI успешно выполнен!")
print("Найдено совпадений:", len(bank_result_new))
print("Время выполнения:", round(elapsed, 2), "сек.")
print("Первые 10 совпадений:", bank_result_new[:10])


Начинаем PSI...
Банк: 307491
Телеком: 307511

PSI успешно выполнен!
Найдено совпадений: 307491
Время выполнения: 114.97 сек.
Первые 10 совпадений: ['332654', '368883', '203457', '103911', '151755', '151618', '149064', '325321', '225699', '160932']


In [ ]:
import private_set_intersection.python as psi
import time

# 1. Меняем роли участников
telecom_client = psi.client.CreateWithNewKey(True)
bank_server = psi.server.CreateWithNewKey(True)

start = time.perf_counter()

# 2. Банк создаёт защищённое представление своей базы
bank_setup = bank_server.CreateSetupMessage(
    0.001,
    len(telecom_new),
    bank_new,
    psi.DataStructure.RAW
)

# 3. Телеком создаёт криптографический запрос
telecom_request = telecom_client.CreateRequest(telecom_new)

# 4. Банк обрабатывает запрос телекома
bank_response = bank_server.ProcessRequest(telecom_request)

# 5. Телеком вычисляет пересечение
telecom_indices = telecom_client.GetIntersection(
    bank_setup,
    bank_response
)

# 6. Восстанавливаем совпавшие ID из базы телекома
telecom_result = [
    telecom_new[i] for i in telecom_indices
]

elapsed_second = time.perf_counter() - start

print("Второй PSI успешно выполнен!")
print("Найдено совпадений:", len(telecom_result))
print(f"Время: {elapsed_second:.2f} сек.")

# 7. Сравниваем результаты двух независимых запусков
assert set(bank_result_new) == set(telecom_result)

# 8. Сравниваем с обычным пересечением
assert set(telecom_result) == expected_new

print("Результаты банка и телекома совпадают!")


Второй PSI успешно выполнен!
Найдено совпадений: 307491
Время: 111.19 сек.
Результаты банка и телекома совпадают!


In [ ]:
# Создаём клиент и сервер для PSI-CA
bank_client = psi.client.CreateWithNewKey(False)
telecom_server = psi.server.CreateWithNewKey(False)

# Телеком подготавливает защищённую базу
server_setup = telecom_server.CreateSetupMessage(
    0.001,
    len(bank_new),
    telecom_new,
    psi.DataStructure.RAW
)

# Банк формирует запрос
request = bank_client.CreateRequest(bank_new)

# Телеком обрабатывает запрос
response = telecom_server.ProcessRequest(request)

# Банк получает только количество совпадений PSI-CA
intersection_size = bank_client.GetIntersectionSize(
    server_setup,
    response
)

print("Количество совпадений:", intersection_size)

Количество совпадений: 307491


In [ ]:
!pip install phe